# 07 · Leakage Experiment: How Much Does a Careless Split Inflate Results?

**Question:** the public Kaggle DFU dataset has duplicate files and many patches cut from the same photo. If a study splits it randomly, how much better do the results *look*?

| Protocol | Data | How folds are made | What it represents |
|---|---|---|---|
| **A. Careless** | All 1,055 files, copies included | Random | A study that never checks for duplicates |
| **B. De-duplicated only** | 713 unique images | Random, ignoring source photos | A study that removes copies, but splits patches of the same photo apart |
| **C. Leakage-free (ours)** | 713 unique images | Grouped by source photo (notebook 03) | Our protocol. Results are reused from notebook 05b |

**Models**
- **MobileNetV3-Small:** our selected backbone.
- **ResNet50:** the most common DFU baseline.
- **Scratch CNN:** not saturated near 99%, so it can show an effect the pretrained models may hide.

**The training recipe is exactly the same as notebook 05b.** Only the split changes.

**Three measurements**
1. **Leakage census (no training needed):** how many test images have a copy or a relative in the training set?
2. **Score inflation:** A and B compared with C. B vs C is the cleanest comparison, because both use the same 713 images and the same training-set size. A also trains on more files.
3. **Error by relationship:** within A and B, are test images with a relative in training easier than those without?

**Honest expectation:** pretrained models already score about 99% on the clean split, so they have little room to inflate. A small difference would be a valid finding, and the notebook reports whatever it measures.

**Accelerator:** GPU T4 x2 · **Internet:** On. Use **Save Version → Save & Run All** (about 2.5-3 hours).
**Inputs:** the dataset, **DFU 03 - Leakage-Free Split** and **DFU 05b - Final Benchmark Results**.
⚠️ **Do not attach DFU 06** (the notebook stops with a clear message if 05b is missing).
**To skip retraining:** also attach an earlier version of this notebook (DFU-07) as input; its finished runs are reused.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest, fisher_exact
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

# ---------------- settings (same recipe as notebook 05b) ----------------
CONFIG = {
    "MODELS": ["MobileNetV3Small", "ResNet50", "ScratchCNN"],
    "PROTOCOLS": ["A", "B"],          # protocol C (leakage-free) is reused from notebook 05b
    "FOLDS": [0, 1, 2, 3, 4],
    "IMG_SIZE": 224, "BATCH": 32,
    "EPOCHS_HEAD": 5, "EPOCHS_FT": 30,
    "LR_HEAD": 1e-3, "LR_FT_OPTIONS": [1e-5, 1e-4],
    "PATIENCE": 7,
    "SCRATCH_EPOCHS": 60, "SCRATCH_LR": 1e-3, "SCRATCH_PATIENCE": 10,
    "SEED": 42,
}
# -------------------------------------------------------------------------
json.dump(CONFIG, open("config.json", "w"), indent=2)
PROT_NAME = {"A": "A. Careless (all files, random)", "B": "B. De-duplicated, random", "C": "C. Leakage-free (ours)"}

splits = pd.read_csv(next(Path("/kaggle/input").rglob("splits.csv")))
# notebook 05b output = the folder whose selection.json names a selected BACKBONE (notebook 06 names an attention variant)
cands = [f.parent for f in Path("/kaggle/input").rglob("selection.json") if "selected_backbone" in json.load(open(f))]
assert cands, "DFU 05b - Final Benchmark Results is not attached (Add Input). Do NOT use DFU 06 here."
prev_dir = cands[0]
print("Files:", len(splits), "| unique images:", int(splits.keep.sum()), "| notebook 05b results from:", prev_dir)

## 1. Build the careless folds

In [ ]:
# Random stratified folds, as a study unaware of the leakage would make them (fixed seed; nothing is tuned on this)
def random_folds(frame):
    f = pd.Series(-1, index=frame.index)
    for k, (_, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=CONFIG["SEED"]).split(frame, frame.label)):
        f.iloc[te] = k
    return f

splits["fold_A"] = random_folds(splits)
splits["fold_B"] = -1
splits.loc[splits.keep, "fold_B"] = random_folds(splits[splits.keep])
splits["fold_C"] = splits["fold"]
splits.to_csv("protocol_folds.csv", index=False)
print(splits[splits.keep].groupby("fold_B").label.agg(images="size", ulcer="sum"))

## 2. Leakage census: what does the training set already contain?
This needs no training. Protocol C should show 0%, confirming notebook 03 once more.

In [ ]:
def census(protocol):
    # For each test image: does the TRAINING set hold an exact copy, or a relative (same source group)?
    d = splits if protocol == "A" else splits[splits.keep]
    col = f"fold_{protocol}"
    out = []
    for k in range(5):
        val = (k + 1) % 5
        test, train = d[d[col] == k], d[(d[col] != k) & (d[col] != val)]
        md5s, groups = set(train.md5), set(train.group)
        rel = np.where(test.md5.isin(md5s), "exact copy in training",
                       np.where(test.group.isin(groups), "related patch in training", "no relative in training"))
        out.append(test.assign(relation=rel, test_fold=k)[["class", "file", "test_fold", "relation", "label"]])
    return pd.concat(out)

CATS = ["exact copy in training", "related patch in training", "no relative in training"]
rel_tables = {p: census(p) for p in ["A", "B", "C"]}
tab = pd.DataFrame({PROT_NAME[p]: rel_tables[p].relation.value_counts().reindex(CATS, fill_value=0) for p in rel_tables}).T
tab["test images"] = tab.sum(axis=1)
tab["% with a copy or relative"] = (100 * (tab[CATS[0]] + tab[CATS[1]]) / tab["test images"]).round(1)
tab.to_csv("leakage_census.csv")
tab

## 3. The training script
This cell only **writes** `train_one.py`. It is the notebook 05b script; only the data selection changes.

In [ ]:
%%writefile train_one.py
# One training run: python train_one.py <model> <protocol> <test_fold>
# Same code as notebook 05 v1; settings come from config.json written by the cell above.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import gc, json, os, random, sys, time, warnings
warnings.filterwarnings("ignore", message=".*shuffle=True.*")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from PIL import Image
from scipy.stats import binomtest
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

PATCHES = next(Path("/kaggle/input").rglob("Patches"))
SPLITS = next(Path("/kaggle/input").rglob("splits.csv"))


globals().update(json.load(open("config.json")))

# ---------------- data ----------------
# ONLY CHANGE from notebook 05b: which rows are used, and which fold column
PROTOCOL = sys.argv[2]
splits = pd.read_csv("protocol_folds.csv")
data = (splits if PROTOCOL == "A" else splits[splits.keep]).reset_index(drop=True)
data["fold"] = data[f"fold_{PROTOCOL}"]

def load(row):
    im = Image.open(PATCHES / row["class"] / row.file).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)

X = np.stack([load(r) for _, r in data.iterrows()]).astype(np.float32)   # raw 0-255 pixels
y = data.label.values.astype(np.float32)                                 # 1 = ulcer, 0 = normal
fold_of = data.fold.values

# ---------------- models and preprocessing ----------------
A = keras.applications
identity = lambda t: t
SPECS = {   # name: (constructor, preprocessing function)
    "MobileNetV3Small": (lambda: A.MobileNetV3Small(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True), identity),
    "MobileNetV3Large": (lambda: A.MobileNetV3Large(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True), identity),
    "MobileNetV2":      (lambda: A.MobileNetV2(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)), A.mobilenet_v2.preprocess_input),
    "EfficientNetB0":   (lambda: A.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)), identity),
    "ResNet50":         (lambda: A.ResNet50(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)), A.resnet50.preprocess_input),
    "DenseNet121":      (lambda: A.DenseNet121(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)), A.densenet.preprocess_input),
}

def head(x):
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    return layers.Dense(1, activation="sigmoid", name="classifier")(x)

def build_pretrained(name):
    base = SPECS[name][0]()
    base.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    model = keras.Model(inp, head(base(inp, training=False)), name=name)
    return model, base

def build_scratch():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1 / 255)(inp)
    for f in [32, 64, 128, 256]:
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization(momentum=0.9)(x); x = layers.ReLU()(x)   # fix: default 0.99 adapts too slowly
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization(momentum=0.9)(x); x = layers.ReLU()(x)   # fix: default 0.99 adapts too slowly
        x = layers.MaxPooling2D()(x)
    return keras.Model(inp, head(x), name="ScratchCNN")

PREP = {name: spec[1] for name, spec in SPECS.items()}
PREP["ScratchCNN"] = identity

# ---------------- one training run ----------------
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15, fill_mode="reflect"),
    layers.RandomZoom(0.1, fill_mode="reflect"),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
], name="augment")

def make_ds(ids, prep, training):
    ds = tf.data.Dataset.from_tensor_slices((X[ids], y[ids]))
    if training:
        ds = ds.shuffle(len(ids), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH)
    if training:   # augment on raw pixels, then apply the model's own preprocessing
        ds = ds.map(lambda a, b: (prep(augment(a, training=True)), b), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda a, b: (prep(a), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=[keras.metrics.AUC(name="auc")])

def choose_threshold(p, t):
    # maximise balanced accuracy on the validation fold; ties -> middle of the tied range
    grid = np.linspace(0.01, 0.99, 99)
    bal = np.array([((p >= g)[t == 1].mean() + (p < g)[t == 0].mean()) / 2 for g in grid])
    return float(np.median(grid[bal == bal.max()]))

def run_one(name, test_fold):
    set_seed(SEED)
    val_fold = (test_fold + 1) % 5
    tr = np.where((fold_of != test_fold) & (fold_of != val_fold))[0]
    va, te = np.where(fold_of == val_fold)[0], np.where(fold_of == test_fold)[0]
    prep = PREP[name]
    train_ds, val_ds, test_ds = make_ds(tr, prep, True), make_ds(va, prep, False), make_ds(te, prep, False)
    cw = {0: len(tr) / (2 * (y[tr] == 0).sum()), 1: len(tr) / (2 * (y[tr] == 1).sum())}
    t0 = time.time()

    if name == "ScratchCNN":
        model = build_scratch()
        compile_model(model, keras.optimizers.schedules.CosineDecay(SCRATCH_LR, SCRATCH_EPOCHS * len(train_ds)))
        stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=SCRATCH_PATIENCE, restore_best_weights=True,
                                             start_from_epoch=10)   # validation loss is noisy in the first epochs
        h = model.fit(train_ds, validation_data=val_ds, epochs=SCRATCH_EPOCHS, class_weight=cw, callbacks=[stop], verbose=0)
        chosen, epochs, lr_chosen, head_loss = "scratch", len(h.history["loss"]), SCRATCH_LR, np.nan
        lr_losses = {lr: np.nan for lr in LR_FT_OPTIONS}   # keep the same columns as the pretrained models
    else:
        model, base = build_pretrained(name)
        compile_model(model, LR_HEAD)                                         # phase 1: head only
        model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=cw, verbose=0)
        head_weights, head_loss = model.get_weights(), model.evaluate(val_ds, verbose=0)[0]
        base.trainable = True                                                 # phase 2: fine-tune, BN frozen
        for l in base.layers:
            if isinstance(l, layers.BatchNormalization):
                l.trainable = False
        best = {"val_loss": head_loss, "weights": head_weights, "chosen": "head-only", "lr": None, "epochs": EPOCHS_HEAD}
        lr_losses = {}
        for lr in LR_FT_OPTIONS:                                              # same two options for every model
            set_seed(SEED)
            model.set_weights(head_weights)                                   # every option starts from the same head
            compile_model(model, keras.optimizers.schedules.CosineDecay(lr, EPOCHS_FT * len(train_ds)))
            stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
            h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=cw, callbacks=[stop], verbose=0)
            lr_losses[lr] = min(h2.history["val_loss"])
            if lr_losses[lr] < best["val_loss"]:
                best = {"val_loss": lr_losses[lr], "weights": model.get_weights(), "chosen": f"fine-tuned",
                        "lr": lr, "epochs": EPOCHS_HEAD + len(h2.history["loss"])}
        model.set_weights(best["weights"])
        chosen, epochs, lr_chosen = best["chosen"], best["epochs"], best["lr"]
    train_min = (time.time() - t0) / 60

    thr = choose_threshold(model.predict(val_ds, verbose=0).ravel(), y[va])
    p = model.predict(test_ds, verbose=0).ravel()
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y[te], pred, labels=[0, 1]).ravel()
    row = {"model": name, "fold": test_fold, "threshold": thr, "chosen": chosen, "lr_ft": lr_chosen, "epochs": epochs,
           "val_loss_head": head_loss, **{f"val_loss_lr{lr:g}": v for lr, v in lr_losses.items()},
           "train_min": round(train_min, 2), "n_test": len(te), "tp": tp, "fn": fn, "tn": tn, "fp": fp,
           "accuracy": accuracy_score(y[te], pred), "precision": precision_score(y[te], pred, zero_division=0),
           "sensitivity": recall_score(y[te], pred), "specificity": tn / (tn + fp),
           "f1": f1_score(y[te], pred), "auc": roc_auc_score(y[te], p)}
    preds = pd.DataFrame({"model": name, "fold": test_fold, "file": data.file[te].values,
                          "class": data["class"][te].values, "label": y[te].astype(int), "prob_ulcer": p, "pred": pred})
    return row, preds, model

# ---------------- size, FLOPs and speed ----------------
def count_flops(model, prep):
    # FLOPs for one 224x224 image, from TensorFlow's graph profiler (multiply-add = 2 FLOPs)
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
        fn = tf.function(lambda t: model(prep(t), training=False))
        conc = fn.get_concrete_function(tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32))
        frozen = convert_variables_to_constants_v2(conc)
        B = tf.compat.v1.profiler.ProfileOptionBuilder
        opts = B(B.float_operation()).with_empty_output().build()   # silence the long profiler printout
        info = tf.compat.v1.profiler.profile(graph=frozen.graph, run_meta=tf.compat.v1.RunMetadata(), cmd="op", options=opts)
        return info.total_float_ops
    except Exception as e:
        print("   FLOPs not available:", type(e).__name__)
        return None

def latency_ms(model, prep, device, runs=100):
    with tf.device(device):
        infer = tf.function(lambda t: model(prep(t), training=False))
        x = tf.constant(X[:1])
        for _ in range(10): infer(x).numpy()
        t = time.perf_counter()
        for _ in range(runs): infer(x).numpy()
        return (time.perf_counter() - t) / runs * 1000

def efficiency(model, name):
    prep = PREP[name]
    return {"model": name, "params_M": model.count_params() / 1e6, "size_mb_fp32": model.count_params() * 4 / 1e6,
            "gflops": (lambda f: f / 1e9 if f else None)(count_flops(model, prep)),
            "latency_gpu_ms": latency_ms(model, prep, "/GPU:0") if tf.config.list_physical_devices("GPU") else None,
            "latency_cpu_ms": latency_ms(model, prep, "/CPU:0", runs=30)}


if __name__ == "__main__":
    name, k = sys.argv[1], int(sys.argv[3])
    RES, PRED = Path("results_per_fold.csv"), Path("test_predictions_all.csv")
    row, preds, model = run_one(name, k)
    row = {"protocol": PROTOCOL, **row}
    preds.insert(0, "protocol", PROTOCOL)
    if RES.exists():
        pd.DataFrame([row]).reindex(columns=pd.read_csv(RES, nrows=0).columns).to_csv(RES, mode="a", header=False, index=False)
    else:
        pd.DataFrame([row]).to_csv(RES, index=False)
    preds.to_csv(PRED, mode="a", header=not PRED.exists(), index=False)
    print(f"RESULT {PROTOCOL} {name:17s} fold {k} | n_test {row['n_test']} | acc {row['accuracy']:.3f}  sens {row['sensitivity']:.3f}  spec {row['specificity']:.3f}"
          f"  F1 {row['f1']:.3f}  AUC {row['auc']:.3f} | thr {row['threshold']:.2f} | {row['chosen']:10s} lr {row['lr_ft']} | {row['train_min']:.1f} min")

## 4. Run protocols A and B
This is the long cell: 30 runs, with a result line after each. Re-running continues where it left off.

In [ ]:
RES, PRED = Path("results_per_fold.csv"), Path("test_predictions_all.csv")
# Resume: if an earlier version of THIS notebook is attached as input, reuse its finished runs instead of retraining
if not RES.exists():
    for f in Path("/kaggle/input").rglob("results_per_fold.csv"):
        if "protocol" in pd.read_csv(f, nrows=0).columns and (f.parent / "test_predictions_all.csv").exists():
            import shutil
            shutil.copy(f, RES); shutil.copy(f.parent / "test_predictions_all.csv", PRED)
            print("Reusing", len(pd.read_csv(RES)), "finished runs from", f.parent)
            break
done = set(map(tuple, pd.read_csv(RES)[["protocol", "model", "fold"]].values)) if RES.exists() else set()
env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="2")

t_start = time.time()
for protocol in CONFIG["PROTOCOLS"]:
    for name in CONFIG["MODELS"]:
        for k in CONFIG["FOLDS"]:
            if (protocol, name, k) in done:
                continue
            for attempt in (1, 2):
                r = subprocess.run([sys.executable, "train_one.py", name, protocol, str(k)], capture_output=True, text=True, env=env)
                result = [l for l in r.stdout.splitlines() if l.startswith("RESULT ")]
                if r.returncode == 0 and result:
                    print(result[-1][7:] + f" | total {(time.time() - t_start) / 60:.0f} min", flush=True)
                    break
                print(f"!! {protocol} {name} fold {k}: attempt {attempt} failed (exit code {r.returncode}). Last error lines:")
                print("\n".join((r.stderr or r.stdout).splitlines()[-15:]), flush=True)
            else:
                raise RuntimeError(f"{protocol} {name} fold {k} failed twice - see the error lines above")
print("All runs finished.")

## 5. Results and inflation
Protocol C is copied unchanged from notebook 05b.

In [ ]:
# Combine: protocols A and B from this notebook + protocol C from notebook 05b (unchanged)
new_res, new_pred = pd.read_csv(RES), pd.read_csv(PRED)
old_res = pd.read_csv(prev_dir / "results_per_fold.csv"); old_pred = pd.read_csv(prev_dir / "test_predictions_all.csv")
models = CONFIG["MODELS"]
old_res = old_res[old_res.model.isin(models)].assign(protocol="C"); old_pred = old_pred[old_pred.model.isin(models)].assign(protocol="C")
all_res = pd.concat([new_res, old_res], ignore_index=True); all_pred = pd.concat([new_pred, old_pred], ignore_index=True)
all_res.to_csv("results_all_protocols.csv", index=False); all_pred.to_csv("predictions_all_protocols.csv", index=False)

metrics = ["accuracy", "sensitivity", "specificity", "f1", "auc"]
g = all_res.groupby(["model", "protocol"])[metrics].agg(["mean", "std"])
summary = pd.DataFrame({m: [f"{100 * g.loc[i, (m, 'mean')]:.2f} ± {100 * g.loc[i, (m, 'std')]:.2f}" for i in g.index] for m in metrics}, index=g.index)
summary = summary.loc[[(m, p) for m in models for p in "ABC"]]
summary.to_csv("summary_by_protocol.csv")
summary

In [ ]:
# Inflation = careless score minus leakage-free score. McNemar compares the SAME images in both protocols
def per_image(model, protocol):
    d = all_pred[(all_pred.model == model) & (all_pred.protocol == protocol)]
    if protocol == "A":     # for a paired test, use one file per unique image
        keep = splits[splits.keep][["class", "file"]]
        d = d.merge(keep, on=["class", "file"])
    return d.set_index(["class", "file"])

rows = []
for m in models:
    c = per_image(m, "C")
    f1c = all_res[(all_res.model == m) & (all_res.protocol == "C")].f1.mean()
    errc = all_pred[(all_pred.model == m) & (all_pred.protocol == "C")]
    for p in "AB":
        d = per_image(m, p)
        idx = c.index.intersection(d.index)
        ok_p, ok_c = d.loc[idx, "pred"] == d.loc[idx, "label"], c.loc[idx, "pred"] == c.loc[idx, "label"]
        b, cc = int((ok_p & ~ok_c).sum()), int((~ok_p & ok_c).sum())       # only this protocol right / only C right
        full = all_pred[(all_pred.model == m) & (all_pred.protocol == p)]
        rows.append({"model": m, "protocol": p,
                     "F1 inflation (points)": round(100 * (all_res[(all_res.model == m) & (all_res.protocol == p)].f1.mean() - f1c), 2),
                     "accuracy as reported (%)": round(100 * (full.pred == full.label).mean(), 2),
                     "accuracy leakage-free (%)": round(100 * (errc.pred == errc.label).mean(), 2),
                     "paired images": len(idx), "only this protocol right": b, "only C right": cc,
                     "McNemar p": round(binomtest(b, b + cc, 0.5).pvalue, 4) if b + cc else 1.0})
inflation = pd.DataFrame(rows).set_index(["model", "protocol"])
inflation.to_csv("inflation_table.csv")
inflation

## 6. Error by relationship

In [ ]:
# Within each careless protocol: are test images with a relative in training easier?
rows = []
for m in models:
    for p in "AB":
        d = all_pred[(all_pred.model == m) & (all_pred.protocol == p)].merge(rel_tables[p][["class", "file", "relation"]], on=["class", "file"])
        d = d.assign(wrong=d.pred != d.label, related=d.relation != "no relative in training")
        a, b, c_, e = int(d[d.related].wrong.sum()), int((~d[d.related].wrong).sum()), int(d[~d.related].wrong.sum()), int((~d[~d.related].wrong).sum())
        row = {"model": m, "protocol": p}
        for cat in CATS:
            s = d[d.relation == cat]
            row[f"{cat}: n"] = len(s); row[f"{cat}: error %"] = round(100 * s.wrong.mean(), 2) if len(s) else None
        row["Fisher p (related vs none)"] = round(fisher_exact([[a, b], [c_, e]])[1], 4) if (a + b) and (c_ + e) else None
        rows.append(row)
by_relation = pd.DataFrame(rows).set_index(["model", "protocol"])
by_relation.to_csv("error_by_relation.csv")
by_relation

## 7. Chart and key numbers

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
x = np.arange(len(models)); w = 0.26
cols = {"A": "#b22222", "B": "#ef8a3c", "C": "#1f3b63"}
for i, p in enumerate("ABC"):
    r = all_res[all_res.protocol == p].groupby("model").f1.agg(["mean", "std"]).loc[models]
    ax[0].bar(x + (i - 1) * w, 100 * r["mean"], w, yerr=100 * r["std"], capsize=3, color=cols[p], label=PROT_NAME[p])
ax[0].set_xticks(x, models); ax[0].set_ylabel("F1 (%), mean ± std over 5 folds"); ax[0].set_title("F1 by split protocol")
ax[0].set_ylim(max(0, 100 * all_res.f1.min() - 3), 101); ax[0].legend(fontsize=8, loc="lower left")
t = tab[CATS]; left = np.zeros(len(t)); shades = ["#b22222", "#ef8a3c", "#1f3b63"]
pct = 100 * t.div(t.sum(axis=1), axis=0)
for cat, col in zip(CATS, shades):
    ax[1].barh(list(pct.index), pct[cat], left=left, color=col, label=cat); left += pct[cat].values
ax[1].invert_yaxis(); ax[1].set_xlabel("% of test images"); ax[1].set_title("What the training set already contains"); ax[1].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig("leakage_chart.png", dpi=130); plt.show()

print("Key numbers (from this run):")
for p in "ABC":
    print(f"  {PROT_NAME[p]}: {tab.loc[PROT_NAME[p], '% with a copy or relative']}% of test images have a copy or relative in training")
for (m, p), r in inflation.iterrows():
    sig = "significant" if r["McNemar p"] < 0.05 else "not significant"
    print(f"  {m}, protocol {p} vs C: F1 inflation {r['F1 inflation (points)']:+.2f} points (McNemar p = {r['McNemar p']}, {sig})")

## How to read the results
- **Section 2 (census)** is the safest finding: it needs no model and shows directly how much a random split leaks.
- **Section 5:** a positive inflation means the careless split looks better. Protocol A also trains on more files, so **B vs C is the fair comparison** for source-photo leakage.
- **Section 6:** if test images with a relative in training have fewer errors, that directly shows leakage making the test easier.
- **Limitation:** one random split per protocol, with one seed.